### Step 4: MLOps Practices

Version Control

• Use GitHub

Experiment Tracking

• Track: Models, Metrics, Parameters

* MLOps means applying software engineering and operational practices to Machine Learning projects.It helps manage models throughout their lifecycle.

Version Control:

*Create Repository # After open vs code terminal

*git init # Initialize git

*git add . # Add files

*git commit -m "Initial ML project" # Commit your changes

*git remote add origin GITHUB_REPOSITORY_URL # Connect GitHub repository.

*git branch -M main

*git push -u origin main # Push the code

In [75]:
!pip install mlflow

  Using cached graphql_core-3.2.14-py3-none-any.whl.metadata (11 kB)
Using cached graphql_core-3.2.14-py3-none-any.whl (279 kB)
  Attempting uninstall: graphql-core
    Found existing installation: graphql-core 3.3.2
    Uninstalling graphql-core-3.3.2:
      Successfully uninstalled graphql-core-3.3.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gql 4.4.0 requires graphql-core<3.4,>=3.3.0a3, but you have graphql-core 3.2.14 which is incompatible.


In [76]:
import mlflow
import mlflow.sklearn

In [77]:
mlflow.set_experiment(
    "Regression_Model_Comparison"
)

2026/11/21 19:48:55 INFO mlflow.tracking.fluent: Experiment with name 'Regression_Model_Comparison' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/0d85beca6d5d4fdc94bffcdd3c8c7264', creation_time=1795270738896, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1795270738896, lifecycle_stage='active', name='Regression_Model_Comparison', tags={}, trace_location=None, workspace='default'>

In [78]:
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_rf_model = random_search.best_estimator_

train_pred = best_rf_model.predict(X_train)
test_pred = best_rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, test_pred)
rf_mse = mean_squared_error(y_test, test_pred)
rf_rmse = np.sqrt(rf_mse)
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

mlflow.set_experiment("Regression_Model_Comparison")

with mlflow.start_run(run_name="Tuned_Random_Forest"):
    # Parameters
    mlflow.log_params(random_search.best_params_)

    # Metrics
    mlflow.log_metric("MAE", rf_mae)
    mlflow.log_metric("MSE", rf_mse)
    mlflow.log_metric("RMSE", rf_rmse)
    mlflow.log_metric("Train_R2", train_r2)
    mlflow.log_metric("Test_R2", test_r2)

    # Log model
    mlflow.sklearn.log_model(
        sk_model=best_rf_model,
        name="random_forest_model",
        serialization_format="cloudpickle",
    )

print("Experiment Tracking Completed!")

2026/11/21 19:49:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Tuned_Random_Forest at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/1/runs/12f41eae696f4af79c6f026fe6024488
🧪 View experiment at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/1
Experiment Tracking Completed!


In [79]:
# Define the dictionary of fitted pipelines before looping
trained_models = {
    "Tuned_Random_Forest": best_rf_model,
    # Add any other fitted pipelines from earlier cells, e.g.:
    # "Baseline_Random_Forest": rf_pipeline,
}

In [80]:
trained_models

{'Tuned_Random_Forest': Pipeline(steps=[('preprocessing',
                  Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                  ('scaler', StandardScaler())])),
                 ('model',
                  RandomForestRegressor(max_depth=30, min_samples_leaf=2,
                                        min_samples_split=5, n_estimators=300,
                                        n_jobs=1, random_state=42))])}

In [81]:
for model_name, pipeline in trained_models.items():
    with mlflow.start_run(run_name=model_name):
        # Log parameters and metrics here...

        # Log trained model as an MLflow artifact
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="trained_model",
            serialization_format="cloudpickle",
        )
        

2026/11/21 19:49:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Tuned_Random_Forest at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/1/runs/c54c400c4dcc40c7b4e4ae2024589b66
🧪 View experiment at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/1


In [82]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline

baseline_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

for name, estimator in baseline_models.items():
    pipe = Pipeline([
        ("preprocessing", create_preprocessor()),
        ("model", estimator),
    ])
    pipe.fit(X_train, y_train)
    trained_models[name] = pipe

In [83]:
models_for_comparison = {
    "Tuned_Random_Forest": best_rf_model,
}

In [84]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

comparison_results = []

for name, pipeline in models_for_comparison.items():

    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)

    comparison_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, test_pred),
        "MSE": mean_squared_error(y_test, test_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
        "Train_R2": r2_score(y_train, train_pred),
        "Test_R2": r2_score(y_test, test_pred)
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    by="RMSE",
    ascending=True
)

print("Total models:", len(trained_models))
print("Model names:", list(trained_models.keys()))

Total models: 4
Model names: ['Tuned_Random_Forest', 'Linear Regression', 'Decision Tree', 'Gradient Boosting']


In [85]:
print(len(trained_models))
print(trained_models.keys())

4
dict_keys(['Tuned_Random_Forest', 'Linear Regression', 'Decision Tree', 'Gradient Boosting'])


In [86]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

comparison_results = []

for model_name, pipeline in trained_models.items():

    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_test_pred)
    mse = mean_squared_error(y_test, y_test_pred)
    rmse = np.sqrt(mse)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    comparison_results.append({
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "Train_R2": train_r2,
        "Test_R2": test_r2
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

print(comparison_df)


                 Model       MAE       MSE      RMSE  Train_R2   Test_R2
0        Decision Tree  0.000000  0.000000  0.000000  1.000000  1.000000
1  Tuned_Random_Forest  0.015483  0.001708  0.041325  0.972925  0.997557
2    Gradient Boosting  0.035992  0.005995  0.077429  0.999171  0.991422
3    Linear Regression  0.146377  0.037114  0.192649  0.925420  0.946896


In [87]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

dt_model = trained_models["Decision Tree"]

y_pred = dt_model.predict(X_test)

print("First 10 Actual Values:")
print(np.array(y_test)[:10])

print("\nFirst 10 Predicted Values:")
print(y_pred[:10])

print("\nAre predictions exactly equal?")
print(np.allclose(np.array(y_test), y_pred))

print("\nUnique predictions:", len(np.unique(y_pred)))
print("Unique actual values:", len(np.unique(y_test)))

First 10 Actual Values:
[1 0 2 1 1 0 1 2 1 1]

First 10 Predicted Values:
[1. 0. 2. 1. 1. 0. 1. 2. 1. 1.]

Are predictions exactly equal?
True

Unique predictions: 3
Unique actual values: 3


In [88]:
!pip install -q dagshub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
graphene 3.4.3 requires graphql-core<3.3,>=3.1, but you have graphql-core 3.3.2 which is incompatible.
graphql-relay 3.2.0 requires graphql-core<3.3,>=3.2, but you have graphql-core 3.3.2 which is incompatible.


In [89]:
import dagshub
import mlflow

dagshub.init(
    repo_owner="nagavedasreethota012-lgtm",
    repo_name="AI-Job-Salary-Prediction",
    mlflow=True
)

mlflow.set_experiment("Salary_Prediction_Experiments")

with mlflow.start_run():
    mlflow.log_param("model", "Random Forest Regressor")
    mlflow.log_metric("test_value", 1.0)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiments:", mlflow.search_experiments())

Initialized MLflow to track repo "nagavedasreethota012-lgtm/AI-Job-Salary-Prediction"

Repository nagavedasreethota012-lgtm/AI-Job-Salary-Prediction initialized!

🏃 View run able-bird-533 at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0/runs/9b2f1a38b10042eb972c7afff573b1f1
🧪 View experiment at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0
Tracking URI: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow
Experiments: [<Experiment: artifact_location='mlflow-artifacts:/0d85beca6d5d4fdc94bffcdd3c8c7264', creation_time=1795270738896, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1795270738896, lifecycle_stage='active', name='Regression_Model_Comparison', tags={}, trace_location=None, workspace='default'>, <Experiment: artifact_location='mlflow-artifacts:/7080b163a4464f6fa247d00649f1e13f', creation_time=1795269497150, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1795269497150, lifecycle_stage='active', name='Salary_Prediction_Experiments', tags={'mlflow.experimentKin

In [90]:
rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)

In [91]:
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 0.014666666666666672
MSE: 0.001493333333333334
RMSE: 0.03864367132317184
R2 Score: 0.9978632750397456


In [92]:
import mlflow

mlflow.set_experiment("Salary_Prediction_Experiments")

with mlflow.start_run(run_name="Random_Forest_Regression"):

    mlflow.log_param("model", "Random Forest Regressor")

    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2_Score", r2)

print("All metrics uploaded to DagsHub!")

🏃 View run Random_Forest_Regression at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0/runs/878df19bcb9e457ead349282e50edc23
🧪 View experiment at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0
All metrics uploaded to DagsHub!


In [93]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline

linear_pipeline = Pipeline([
    ("preprocessing", create_preprocessor()),
    ("model", LinearRegression()),
])

dt_pipeline = Pipeline([
    ("preprocessing", create_preprocessor()),
    ("model", DecisionTreeRegressor(random_state=42)),
])

gb_pipeline = Pipeline([
    ("preprocessing", create_preprocessor()),
    ("model", GradientBoostingRegressor(random_state=42)),
])

models = {
    "Linear Regression": linear_pipeline,
    "Decision Tree": dt_pipeline,
    "Random Forest": rf_pipeline,
    "Gradient Boosting": gb_pipeline,
}

In [94]:
import mlflow
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mlflow.set_experiment("Salary_Prediction_Experiments")
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", name)
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("MSE", mse)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2_Score", r2)

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2,
    })

print("All models logged successfully!")

🏃 View run Linear Regression at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0/runs/4eceb366c77d4dfab5b20786daaa1164
🧪 View experiment at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0
🏃 View run Decision Tree at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0/runs/fb82507ec84049ea8a516036ae4039ef
🧪 View experiment at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0
🏃 View run Random Forest at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0/runs/884df651cc5647e39c5f1ddcbed6ec35
🧪 View experiment at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0
🏃 View run Gradient Boosting at: https://dagshub.com/nagavedasreethota012-lgtm/AI-Job-Salary-Prediction.mlflow/#/experiments/0/runs/0ac4d287503f434eab67a2a8ac1da326


In [95]:
import pandas as pd

comparison_df = pd.DataFrame(results)

print(comparison_df)

               Model       MAE       MSE      RMSE  R2 Score
0  Linear Regression  0.146377  0.037114  0.192649  0.946896
1      Decision Tree  0.000000  0.000000  0.000000  1.000000
2      Random Forest  0.014667  0.001493  0.038644  0.997863
3  Gradient Boosting  0.035992  0.005995  0.077429  0.991422


### Logging & Monitoring

• Log predictions

• Monitor model performance

In [96]:
best_model = comparison_df.loc[
    comparison_df["R2 Score"].idxmax()
]

print("Best Model:")
print(best_model)

Best Model:
Model       Decision Tree
MAE                   0.0
MSE                   0.0
RMSE                  0.0
R2 Score              1.0
Name: 1, dtype: object


In [97]:
best_model_name = comparison_df.loc[
    comparison_df["R2 Score"].idxmax(), "Model"
]

best_model = models[best_model_name]

print("Best Model:", best_model_name)

print("\nBest Model Performance:")
print(comparison_df[
    comparison_df["Model"] == best_model_name
])

Best Model: Decision Tree

Best Model Performance:
           Model  MAE  MSE  RMSE  R2 Score
1  Decision Tree  0.0  0.0   0.0       1.0


In [99]:
mlflow.sklearn.log_model(
    sk_model=best_model,
    name="best_model",
    serialization_format="cloudpickle",
)

2026/11/21 19:51:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
